# Advanced Poisson Regression Topics\n\n**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**\n\n## Learning Objectives\n\n- Understand overdispersion in count data\n- Implement Negative Binomial regression\n- Handle zero-inflation in soccer data\n- Build hurdle models for two-stage prediction\n- Diagnose and fix Poisson model violations\n- Choose the appropriate count model for your data

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport statsmodels.api as sm\nimport statsmodels.formula.api as smf\nfrom scipy import stats\n\nsns.set_style('whitegrid')\nplt.rcParams['figure.figsize'] = (12, 8)

## The Problem: When Poisson Isn't Enough\n\n**Standard Poisson assumption:** Variance = Mean\n\n**Reality in soccer:**\n- Some teams are consistently high/low scoring (extra variance)\n- Many matches end 0-0 (zero-inflation)\n- Goals come in clusters (overdispersion)\n\n**Solutions:**\n1. **Negative Binomial** - Handles overdispersion\n2. **Zero-Inflated Poisson** - Handles excess zeros\n3. **Hurdle Models** - Two-stage process

## Overdispersion: When Variance Exceeds the Mean\n\n**Overdispersion** occurs when the variance in the data is greater than what the Poisson model expects.

In [ ]:
# Generate overdispersed data\nnp.random.seed(42)\nn = 200\n\n# Simulate match data with overdispersion\nshots = np.random.randint(5, 15, n)\n# Add team-level random effects (causes overdispersion)\nteam_effect = np.random.choice([-0.5, 0, 0.5], n)\ngoals_overdispersed = np.random.poisson(shots * 0.15 + team_effect + 1)\n\ndf_over = pd.DataFrame({\n    'Shots': shots,\n    'Goals': goals_overdispersed\n})\n\nprint(\"Overdispersion Check:\")\nprint(f\"Mean goals: {df_over['Goals'].mean():.3f}\")\nprint(f\"Variance: {df_over['Goals'].var():.3f}\")\nprint(f\"Variance/Mean ratio: {df_over['Goals'].var() / df_over['Goals'].mean():.3f}\")\nprint(f\"\\nIf ratio > 1: Overdispersion present!\")\nprint(f\"If ratio ≈ 1: Poisson is appropriate\")\nprint(f\"If ratio < 1: Underdispersion (rare)\")

## Compare Poisson vs. Negative Binomial

In [ ]:
# Fit Poisson model\npoisson_model = smf.poisson('Goals ~ Shots', data=df_over).fit()\n\n# Fit Negative Binomial model\nnb_model = smf.negativebinomial('Goals ~ Shots', data=df_over).fit()\n\nprint(\"Model Comparison:\")\nprint(f\"\\nPoisson AIC: {poisson_model.aic:.2f}\")\nprint(f\"Negative Binomial AIC: {nb_model.aic:.2f}\")\nprint(f\"\\nLower AIC = Better model\")\n\nif nb_model.aic < poisson_model.aic:\n    print(\"✅ Negative Binomial is better - overdispersion confirmed!\")\nelse:\n    print(\"✅ Poisson is sufficient - no significant overdispersion\")\n\nprint(f\"\\nNegative Binomial alpha parameter: {nb_model.params['alpha']:.3f}\")\nprint(\"(Alpha > 0 indicates overdispersion)\")

## Visualize the Difference

In [ ]:
# Get predictions\ndf_over['poisson_pred'] = poisson_model.predict(df_over)\ndf_over['nb_pred'] = nb_model.predict(df_over)\n\n# Plot\nfig, axes = plt.subplots(1, 2, figsize=(14, 6))\n\n# Actual vs Predicted - Poisson\naxes[0].scatter(df_over['Goals'], df_over['poisson_pred'], alpha=0.5)\naxes[0].plot([0, 8], [0, 8], 'r--', linewidth=2)\naxes[0].set_xlabel('Actual Goals')\naxes[0].set_ylabel('Predicted Goals')\naxes[0].set_title('Poisson Regression')\naxes[0].grid(True, alpha=0.3)\n\n# Actual vs Predicted - Negative Binomial\naxes[1].scatter(df_over['Goals'], df_over['nb_pred'], alpha=0.5, color='green')\naxes[1].plot([0, 8], [0, 8], 'r--', linewidth=2)\naxes[1].set_xlabel('Actual Goals')\naxes[1].set_ylabel('Predicted Goals')\naxes[1].set_title('Negative Binomial Regression')\naxes[1].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()\n\nprint(\"Negative Binomial typically provides wider prediction intervals,\")\nprint(\"accounting for the extra variability in the data.\")

## Zero-Inflation: Too Many Zeros\n\n**Problem:** Some matches have more 0-0 draws than a Poisson model would predict.\n\n**Causes in soccer:**\n- Defensive tactics\n- Bad weather\n- Low-stakes matches\n- Mismatched teams playing conservatively

In [ ]:
# Generate zero-inflated data\nnp.random.seed(42)\nn = 200\n\n# Some matches are \"structural zeros\" (defensive matches)\ndefensive_match = np.random.binomial(1, 0.3, n)  # 30% defensive\n\nshots = np.random.randint(5, 15, n)\ngoals_zi = np.where(\n    defensive_match == 1,\n    0,  # Defensive matches always 0 goals\n    np.random.poisson(shots * 0.15)  # Normal matches\n)\n\ndf_zi = pd.DataFrame({\n    'Shots': shots,\n    'Goals': goals_zi\n})\n\nprint(\"Zero-Inflation Check:\")\nprint(f\"Observed zeros: {(df_zi['Goals'] == 0).sum()} ({(df_zi['Goals'] == 0).mean()*100:.1f}%)\")\n\n# Expected zeros under Poisson\nmean_goals = df_zi['Goals'].mean()\nexpected_zeros = len(df_zi) * np.exp(-mean_goals)\nprint(f\"Expected zeros (Poisson): {expected_zeros:.0f} ({expected_zeros/len(df_zi)*100:.1f}%)\")\n\nif (df_zi['Goals'] == 0).sum() > expected_zeros * 1.2:\n    print(\"\\n⚠️ Zero-inflation detected! Consider Zero-Inflated Poisson (ZIP) model.\")\nelse:\n    print(\"\\n✅ No significant zero-inflation.\")

## Visualize Zero-Inflation

In [ ]:
# Compare observed vs expected distribution\nfig, ax = plt.subplots(figsize=(10, 6))\n\n# Observed distribution\nobserved_counts = df_zi['Goals'].value_counts().sort_index()\nax.bar(observed_counts.index - 0.2, observed_counts.values, width=0.4, \n       label='Observed', alpha=0.7, color='blue')\n\n# Expected Poisson distribution\nmax_goals = df_zi['Goals'].max()\nexpected_counts = [len(df_zi) * stats.poisson.pmf(k, mean_goals) for k in range(max_goals + 1)]\nax.bar(np.arange(max_goals + 1) + 0.2, expected_counts, width=0.4, \n       label='Expected (Poisson)', alpha=0.7, color='orange')\n\nax.set_xlabel('Number of Goals')\nax.set_ylabel('Frequency')\nax.set_title('Observed vs. Expected Distribution (Zero-Inflation)')\nax.legend()\nax.grid(True, alpha=0.3, axis='y')\nplt.tight_layout()\nplt.show()\n\nprint(\"Notice the excess of zeros in the observed data (blue bar at 0).\")

## Hurdle Models: Two-Stage Approach\n\n**Idea:** Model the process in two stages:\n1. **Stage 1:** Will any goals be scored? (Binary: Yes/No)\n2. **Stage 2:** If yes, how many? (Count model for positive values)\n\n**When to use:** When the process of scoring the first goal is fundamentally different from scoring additional goals.

In [ ]:
# Implement a simple hurdle model\nfrom sklearn.linear_model import LogisticRegression\n\n# Stage 1: Binary model (any goals?)
df_zi['any_goals'] = (df_zi['Goals'] > 0).astype(int)\nX = df_zi[['Shots']]\n\nlogit_model = LogisticRegression()\nlogit_model.fit(X, df_zi['any_goals'])\nprob_any_goals = logit_model.predict_proba(X)[:, 1]\n\nprint(\"Stage 1: Probability of scoring any goals\")\nprint(f\"Average probability: {prob_any_goals.mean():.3f}\")\n\n# Stage 2: Count model for positive goals\ndf_positive = df_zi[df_zi['Goals'] > 0].copy()\nif len(df_positive) > 0:\n    poisson_positive = smf.poisson('Goals ~ Shots', data=df_positive).fit()\n    print(f\"\\nStage 2: Poisson model for positive goals\")\n    print(f\"Coefficient for Shots: {poisson_positive.params['Shots']:.4f}\")\n    \n    # Combined prediction\n    positive_pred = poisson_positive.predict(df_zi[['Shots']])\n    hurdle_pred = prob_any_goals * positive_pred\n    \n    print(f\"\\nHurdle model combines both stages:\")\n    print(f\"Prediction = P(any goals) × E(goals | goals > 0)\")\nelse:\n    print(\"Not enough positive observations for stage 2.\")

## Goodness-of-Fit Tests\n\nHow do we know if our Poisson model fits well?

In [ ]:
# Pearson chi-square test for Poisson\npoisson_simple = smf.poisson('Goals ~ Shots', data=df_over).fit()\n\n# Calculate Pearson chi-square statistic\nresiduals = poisson_simple.resid_pearson\nchi_square = np.sum(residuals**2)\ndf_resid = len(df_over) - len(poisson_simple.params)\n\nprint(\"Goodness-of-Fit Test:\")\nprint(f\"Pearson chi-square: {chi_square:.2f}\")\nprint(f\"Degrees of freedom: {df_resid}\")\nprint(f\"Ratio (chi²/df): {chi_square/df_resid:.2f}\")\nprint(f\"\\nInterpretation:\")\nprint(f\"  Ratio ≈ 1: Good fit\")\nprint(f\"  Ratio > 1: Overdispersion (consider Negative Binomial)\")\nprint(f\"  Ratio < 1: Underdispersion (rare)\")\n\nif chi_square/df_resid > 1.5:\n    print(f\"\\n⚠️ Model shows overdispersion. Consider Negative Binomial regression.\")\nelif chi_square/df_resid < 0.7:\n    print(f\"\\n⚠️ Model shows underdispersion. Check for model misspecification.\")\nelse:\n    print(f\"\\n✅ Poisson model fits reasonably well.\")

## Model Selection Guide\n\n| Situation | Recommended Model |\n|-----------|------------------|\n| **Variance ≈ Mean** | Standard Poisson |\n| **Variance > Mean** | Negative Binomial |\n| **Excess zeros** | Zero-Inflated Poisson (ZIP) |\n| **Two-stage process** | Hurdle Model |\n| **Overdispersion + zeros** | Zero-Inflated Negative Binomial |\n\n**Decision Tree:**\n1. Check variance/mean ratio → If > 1.5, consider Negative Binomial\n2. Check zero proportion → If excess, consider ZIP or Hurdle\n3. Compare models using AIC/BIC → Lower is better\n4. Validate with holdout data → Test generalization

## Practical Example: Choosing the Right Model

In [ ]:
# Generate realistic soccer data\nnp.random.seed(42)\nn = 300\n\nshots = np.random.randint(3, 18, n)\npossession = np.random.uniform(35, 65, n)\n\n# Realistic goal generation with overdispersion\nteam_quality = np.random.choice([0.8, 1.0, 1.2], n)\ngoals = np.random.negative_binomial(\n    n=2,  # dispersion parameter\n    p=2/(2 + shots * 0.12 * team_quality)\n)\n\ndf_real = pd.DataFrame({\n    'Shots': shots,\n    'Possession': possession,\n    'Goals': goals\n})\n\nprint(\"Realistic Soccer Data Analysis:\")\nprint(f\"Mean: {df_real['Goals'].mean():.3f}\")\nprint(f\"Variance: {df_real['Goals'].var():.3f}\")\nprint(f\"Variance/Mean: {df_real['Goals'].var()/df_real['Goals'].mean():.3f}\")\nprint(f\"Zero proportion: {(df_real['Goals']==0).mean()*100:.1f}%\")\n\n# Fit multiple models\nmodels = {}\nmodels['Poisson'] = smf.poisson('Goals ~ Shots + Possession', data=df_real).fit()\nmodels['NegBinom'] = smf.negativebinomial('Goals ~ Shots + Possession', data=df_real).fit()\n\nprint(f\"\\nModel Comparison:\")\nfor name, model in models.items():\n    print(f\"  {name}: AIC = {model.aic:.2f}\")\n\nbest_model = min(models.items(), key=lambda x: x[1].aic)\nprint(f\"\\n✅ Best model: {best_model[0]}\")

## Summary\n\nIn this notebook, we:\n\n1. ✅ Understood overdispersion and its causes\n2. ✅ Implemented Negative Binomial regression\n3. ✅ Detected and handled zero-inflation\n4. ✅ Built hurdle models for two-stage processes\n5. ✅ Performed goodness-of-fit tests\n6. ✅ Learned model selection criteria\n\n## Key Takeaways\n\n- **Standard Poisson** assumes variance = mean (often violated in soccer)\n- **Negative Binomial** handles overdispersion (variance > mean)\n- **Zero-inflation** is common in soccer (defensive matches)\n- **Hurdle models** separate \"any goals\" from \"how many\"\n- **Always check assumptions** before choosing a model\n- **AIC/BIC** help compare non-nested models\n- **Real soccer data** often requires Negative Binomial\n\n## When to Use Each Model\n\n- **Poisson:** Quick baseline, well-behaved data\n- **Negative Binomial:** Most soccer applications (handles overdispersion)\n- **ZIP/Hurdle:** Defensive leagues, cup competitions with many 0-0 draws\n- **ZINB:** Combination of overdispersion and zero-inflation

## Practice Exercises\n\n1. **Implement ZIP Model**: Use `statsmodels` to fit a Zero-Inflated Poisson model\n2. **Compare All Models**: Fit Poisson, NB, ZIP, and Hurdle on the same data\n3. **Simulate Data**: Generate data with known properties and verify model selection\n4. **Real Match Data**: Apply to actual league data and identify the best model\n5. **Prediction Intervals**: Generate prediction intervals for count models\n6. **Temporal Effects**: Add time trends to account for changing goal-scoring patterns